Step 1: Imports

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [2]:
# Load environment variables
load_dotenv()

True

Step 2: Define a Pydantic Model

In [3]:
# Define a Pydantic model representing the output
class CapitalInfo(BaseModel):
    
    country: str = Field(description='The name of the country')
    capital: str = Field(description='The capital of the country')
    population: int = Field(description='Approximate population of the capital city')
    
# Print the model
print('Pydantic model defined:')
print(CapitalInfo)

Pydantic model defined:
<class '__main__.CapitalInfo'>


Step 3: Create the Parser and Get Format Instructions

In [4]:
# Create the PydanticOutputParser from the CapitalInfo model
parser = PydanticOutputParser(pydantic_object=CapitalInfo)

# Get the format instructions string: tells the model how to output JSON
format_instructions = parser.get_format_instructions()

print('Format instructions:')
print(format_instructions)

Format instructions:
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"country": {"description": "The name of the country", "title": "Country", "type": "string"}, "capital": {"description": "The capital of the country", "title": "Capital", "type": "string"}, "population": {"description": "Approximate population of the capital city", "title": "Population", "type": "integer"}}, "required": ["country", "capital", "population"]}
```


Step 4: Build Prompt and Chain

In [5]:
# Create the chat model
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Build a prompt that includes the format instructions from the parser
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human', 'Provide information about the country {country_name}.\n\n{format_instructions}')
])

# Use the parser's format instructions in the prompt by partialing them
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# Build the chain: prompt -> model -> parser (Pydantic object)
chain = prompt | llm | parser

# Invoke the chain with a country name

res = chain.invoke({'country_name': 'Nigeria'})

print('Result:')
print(res)
print('Type:', type(res))

Result:
country='Nigeria' capital='Abuja' population=12358858
Type: <class '__main__.CapitalInfo'>


Step 5: Access Pydantic Object Fields

In [7]:
# Step 5: Access Pydantic Object Fields
print(f'Country: {res.country}')
print(f'Capital: {res.capital}')
print(f'Population: {res.population}')

# convert it to a dictionary
print(f'\nAs dictionary: {res.model_dump()}')

Country: Nigeria
Capital: Abuja
Population: 12358858

As dictionary: {'country': 'Nigeria', 'capital': 'Abuja', 'population': 12358858}


<div style="font-size: 0.85em;">
  <h3>PydanticOutputParser – Summary</h3>
  <ul>
    <li><strong>What it does:</strong> Converts the model’s JSON output into a <strong>typed Pydantic object</strong> with validated fields.</li>
    <li><strong>Why use it:</strong> Gives you structured, reliable data with type checking, instead of a raw string or dict.</li>
    <li><strong>How it works:</strong>
      <ol>
        <li>Define a Pydantic model with fields and descriptions.</li>
        <li>Create a <code>PydanticOutputParser</code> from that model.</li>
        <li>Get format instructions and put them into the prompt.</li>
        <li>The model returns JSON that matches the schema.</li>
        <li>The parser validates and returns a Pydantic object.</li>
      </ol>
    </li>
  </ul>
</div>